In [8]:
import random
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import os

# ------------------------------------------------------------
# Random seeds
# ------------------------------------------------------------
np.random.seed(42)
random.seed(42)

# ------------------------------------------------------------
# 1. Sets & basic parameters
# ------------------------------------------------------------
numberCourses = 25      
numberRooms = 10        
numberSlots = 24       
numberDays = 4
slotsPerDay = 6
numberStudents = 400    # Scaled student pool for realistic conflict resolution

courses = [f"CS_{100 + i*10}" for i in range(numberCourses)]
rooms = [f"Room_{101 + i}" for i in range(numberRooms)]
timeSlots = [f"Slot_{t+1}" for t in range(numberSlots)]

dayOfSlot = {timeSlots[t]: t // slotsPerDay + 1 for t in range(numberSlots)}

# ------------------------------------------------------------
# 2. Course enrollments & room capacities
# ------------------------------------------------------------
courseEnrollments = {c: int(np.random.randint(25, 90)) for c in courses} 

# Generous room capacities to avoid room-size bottlenecks
capacityList = [60, 80, 100, 120, 140, 160, 180, 200, 220, 250] 
roomCapacities = {rooms[i]: capacityList[i] for i in range(numberRooms)}

# ------------------------------------------------------------
# 3. Simulate student and course enrollments
# ------------------------------------------------------------
studentID = [f"S_{1000 + i}" for i in range(numberStudents)]
enrollmentMatrix = np.zeros((numberStudents, numberCourses), dtype=int)

for s_idx, s in enumerate(studentID):
    enrolled_course_indices = np.random.choice(numberCourses, size=np.random.randint(3, 5), replace=False)
    enrollmentMatrix[s_idx, enrolled_course_indices] = 1

studentCourseDataFrame = pd.DataFrame(enrollmentMatrix, index=studentID, columns=courses)

# Adjust course enrollments to match actual number of students per course
for j, c in enumerate(courses):
    courseEnrollments[c] = int(studentCourseDataFrame[c].sum())

# ------------------------------------------------------------
# 4. Course to course conflict parameter
# ------------------------------------------------------------
overlap_dict = {}
for i, c1 in enumerate(courses):
    for j, c2 in enumerate(courses):
        if i < j:
            shared_students = np.sum(studentCourseDataFrame[c1] & studentCourseDataFrame[c2])
            overlap_dict[(c1, c2)] = int(shared_students)

print("--- Data Simulation Complete ---")
print(f"Courses: {numberCourses} | Rooms: {numberRooms} | Time Slots: {numberSlots} | Students: {numberStudents}")


# ============================================================
# 5. OPTIMIZATION MODEL (Translating Catherine's Constraints)
# ============================================================

model = gp.Model("University_Exam_Scheduling")

# --- Decision Variables ---
x = model.addVars(courses, rooms, timeSlots, vtype=GRB.BINARY, name="x")

# --- Objective Function ---
model.setObjective(0, GRB.MINIMIZE)

# --- Constraints ---
# Constraint 1: Each course gets exactly one time slot and room assignment
model.addConstrs(
    (gp.quicksum(x[c, r, t] for r in rooms for t in timeSlots) == 1 for c in courses),
    name="OneAssignmentPerCourse"
)

# Constraint 2: Room capacity must be greater than or equal to students enrolled in course
model.addConstrs(
    (courseEnrollments[c] * x[c, r, t] <= roomCapacities[r] 
     for c in courses for r in rooms for t in timeSlots),
    name="RoomCapacity"
)

# Constraint 3: The classroom must be empty during time slot (This means at most 1 exam per room per slot)
model.addConstrs(
    (gp.quicksum(x[c, r, t] for c in courses) <= 1 for r in rooms for t in timeSlots),
    name="RoomMutex"
)

# Constraint 4: Student cannot already have an exam scheduled during time t
print("Building student conflict constraints...")
for s in studentID:
    enrolled_courses = studentCourseDataFrame.columns[studentCourseDataFrame.loc[s] == 1].tolist()
    if enrolled_courses:
        for t in timeSlots:
            model.addConstr(
                gp.quicksum(x[c, r, t] for c in enrolled_courses for r in rooms) <= 1,
                name=f"StudentConflict_{s}_{t}"
            )

# ------------------------------------------------------------
# 6. Solve Model
# ------------------------------------------------------------
print("Optimizing model with Gurobi...")
model.optimize()

# ------------------------------------------------------------
# 7. Export Results & Outputs
# ------------------------------------------------------------
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

studentCourseDataFrame.to_csv("data/student_enrollments.csv")
conflictDataFrame = pd.DataFrame(
    [(c1, c2, count) for (c1, c2), count in overlap_dict.items()],
    columns=["Course 1", "Course 2", "Shared Students"]
)
conflictDataFrame.to_csv("data/course_conflicts.csv", index=False)

if model.status == GRB.OPTIMAL:
    print("\nModel solved successfully to OPTIMAL status!")
    
    schedule_records = []
    for c in courses:
        for r in rooms:
            for t in timeSlots:
                if x[c, r, t].X > 0.5:
                    schedule_records.append({
                        "Course": c,
                        "Enrollment": courseEnrollments[c],
                        "Assigned_Room": r,
                        "Room_Capacity": roomCapacities[r],
                        "Time_Slot": t,
                        "Day": dayOfSlot[t]
                    })
                    
    df_schedule = pd.DataFrame(schedule_records)
    df_schedule.to_csv("outputs/optimal_exam_schedule.csv", index=False)
    print("Optimal exam schedule exported to 'outputs/optimal_exam_schedule.csv'!")
    print(df_schedule.head(10))
else:
    print(f"\nOptimization finished with status {model.status}.")

--- Data Simulation Complete ---
Courses: 25 | Rooms: 10 | Time Slots: 24 | Students: 400
Building student conflict constraints...
Optimizing model with Gurobi...
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 25.2.0 25C56)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 15865 rows, 6000 columns and 349440 nonzeros (Min)
Model fingerprint: 0x93e453eb
Model has 0 linear objective coefficients
Variable types: 0 continuous, 6000 integer (6000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+02]

Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000